# Build an End-to-End System

This puts together the chain of prompts that you saw throughout the course.

## Setup
#### Load the API key and relevant Python libaries.
In this course, we've provided some code that loads the OpenAI API key for you.

In [1]:
!pip install groq
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [2]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


System of chained prompts for processing the user query

===

This function is a complete AI Customer Support Pipeline. It doesn't just answer a user's question—it performs multiple validation and quality checks before returning a response. This kind of workflow is commonly used in production AI chatbot systems.

In [4]:
def evaluate_response(user_input, final_response):

    system_message = """
You are an AI evaluator.

Return only:

Y

or

N

Return Y if the response answers the customer's question.

Otherwise return N.
"""

    user_message = f"""
Customer Question:
{user_input}

Agent Response:
{final_response}
"""

    messages = [
        {"role":"system","content":system_message},
        {"role":"user","content":user_message}
    ]

    return get_completion_from_messages(
        messages,
        temperature=0,
        max_tokens=5
    )

Groq Version of process_user_message()

In [7]:
class Utils:
    def get_products_and_category(self):
        # Placeholder for actual product data
        return {
            "smartx pro phone": {"category": "Smartphones", "description": "A high-performance smartphone."},
            "fotosnap camera": {"category": "Cameras", "description": "A professional DSLR camera."},
            "tv": {"category": "Televisions", "description": "Various models of smart TVs."}
        }

    def find_category_and_product_only(self, user_input, products_data):
        found_products = []
        user_input_lower = user_input.lower()
        for product_name, product_info in products_data.items():
            if product_name in user_input_lower:
                found_products.append(f"Category: {product_info['category']}, Product: {product_name}")
        # Return a string that can be parsed by read_string_to_list
        return "; ".join(found_products)

    def read_string_to_list(self, input_string):
        if not input_string:
            return []
        return [item.strip() for item in input_string.split(';') if item.strip()]

    def generate_output_string(self, product_list):
        output_lines = []
        for item in product_list:
            # Example: item looks like 'Category: Smartphones, Product: smartx pro phone'
            parts = item.split(', ')
            category = parts[0].split(': ')[1]
            product = parts[1].split(': ')[1]
            products_data = self.get_products_and_category()
            description = products_data.get(product, {}).get('description', 'No description available.')
            output_lines.append(f"Product: {product}, Category: {category}, Description: {description}")
        return "\n".join(output_lines)

utils = Utils()


In [8]:
def process_user_message(user_input, all_messages, debug=True):

    delimiter = "```"

    # -----------------------
    # Step 1
    # Find Products
    # -----------------------

    category_and_product_response = utils.find_category_and_product_only(
        user_input,
        utils.get_products_and_category()
    )

    category_and_product_list = utils.read_string_to_list(
        category_and_product_response
    )

    if debug:
        print("Step 1 : Products Extracted")

    # -----------------------
    # Step 2
    # Lookup Product Info
    # -----------------------

    product_information = utils.generate_output_string(
        category_and_product_list
    )

    if debug:
        print("Step 2 : Product Information Retrieved")

    # -----------------------
    # Step 3
    # Generate Response
    # -----------------------

    system_message = """
You are a customer service assistant for a large electronics store.

Answer only using the provided product information.

Be friendly.

Be concise.

Ask relevant follow-up questions.
"""

    messages = [

        {
            "role":"system",
            "content":system_message
        },

        {
            "role":"user",
            "content":f"{delimiter}{user_input}{delimiter}"
        },

        {
            "role":"assistant",
            "content":f"Relevant Product Information:\n{product_information}"
        }

    ]

    final_response = get_completion_from_messages(
        all_messages + messages
    )

    if debug:
        print("Step 3 : Response Generated")

    all_messages = all_messages + messages[1:]

    # -----------------------
    # Step 4
    # Evaluate Response
    # -----------------------

    evaluation = evaluate_response(
        user_input,
        final_response
    )

    if debug:
        print("Step 4 : Response Evaluated")

    # -----------------------
    # Step 5
    # Final Decision
    # -----------------------

    if "Y" in evaluation:

        if debug:
            print("Step 5 : Approved")

        return final_response, all_messages

    else:

        if debug:
            print("Step 5 : Escalated")

        return (
            "I'm unable to answer this correctly. I'll connect you with a human representative.",
            all_messages
        )

Test

In [12]:
user_input = """
tell me about the smartx pro phone and the fotosnap camera,
the dslr one.
Also tell me about your tvs.
"""

response, history = process_user_message(
    user_input,
    []
)

print(response)

Step 1 : Products Extracted
Step 2 : Product Information Retrieved
Step 3 : Response Generated
Step 4 : Response Evaluated
Step 5 : Approved
 

We have the SmartX Pro phone, which is a high-performance smartphone. We also have the Fotosnap camera, a professional DSLR camera. As for our TVs, we offer a range of smart TV models. 

Can you tell me which specific feature of the SmartX Pro phone or the Fotosnap camera interests you? Or are you looking for a particular size or type of TV?


This code is the main execution block of the chatbot. It sends the customer's question to the process_user_message() function, processes the request through the chatbot pipeline, and prints the final response.

| Step | Code                                   | Purpose                                                           |
| ---- | -------------------------------------- | ----------------------------------------------------------------- |
| 1    | `user_input = """..."""`               | Stores the customer's question.                                   |
| 2    | `process_user_message(user_input, [])` | Sends the question to the chatbot processing function.            |
| 3    | `response, history = ...`              | Receives the chatbot's response and updated conversation history. |
| 4    | `print(response)`                      | Displays the final answer to the customer.                        |


| Variable                 | Purpose                                              |
| ------------------------ | ---------------------------------------------------- |
| `user_input`             | Customer's query.                                    |
| `response`               | Final AI-generated response.                         |
| `history`                | Stores conversation history for future interactions. |
| `process_user_message()` | Main chatbot processing function.                    |


What Happens Internally

| Stage                  | Action                                                        |
| ---------------------- | ------------------------------------------------------------- |
| Receive Question       | Accepts the customer's query.                                 |
| Product Detection      | Finds SmartX ProPhone, FotoSnap DSLR Camera, and TV products. |
| Product Lookup         | Retrieves specifications from the product database.           |
| AI Response Generation | Creates a friendly response using the retrieved information.  |
| Response Validation    | Checks whether the response answers the customer's question.  |
| Return Result          | Sends the final response back to the customer.                |


Function that collects user and assistant messages over time

In [10]:
def collect_messages(debug=False):
    user_input = inp.value_input
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context
    #response, context = process_user_message(user_input, context, utils.get_products_and_category(),debug=True)
    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)

This function is the User Interface (UI) Controller of the chatbot. It receives the user's input from the text box, sends it to the chatbot engine, stores the conversation history, and updates the chat interface with both the user's message and the AI's response.

===

| Step | Code                           | Purpose                                                     |
| ---- | ------------------------------ | ----------------------------------------------------------- |
| 1    | `user_input = inp.value_input` | Reads the user's message from the input box.                |
| 2    | `if user_input == ""`          | Prevents processing an empty message.                       |
| 3    | `inp.value = ''`               | Clears the input box after sending the message.             |
| 4    | `process_user_message()`       | Sends the message to the chatbot engine for processing.     |
| 5    | `context.append()`             | Saves the AI response in the conversation history.          |
| 6    | `panels.append()`              | Displays the user's message in the chat window.             |
| 7    | `panels.append()`              | Displays the AI's response in the chat window.              |
| 8    | `return pn.Column(*panels)`    | Refreshes the chat interface with the updated conversation. |

===

| Variable      | Purpose                                                |
| ------------- | ------------------------------------------------------ |
| `user_input`  | Stores the customer's question.                        |
| `inp`         | Text input widget (Panel).                             |
| `context`     | Stores the conversation history.                       |
| `response`    | AI-generated response.                                 |
| `panels`      | List containing all chat messages displayed in the UI. |
| `pn.Column()` | Displays the chat conversation vertically.             |

====

Input Processing

| Code                  | Purpose                                     |
| --------------------- | ------------------------------------------- |
| `inp.value_input`     | Reads the current user input.               |
| `if user_input == ""` | Stops processing if no message was entered. |
| `inp.value = ""`      | Clears the text box after submission.       |


===

Chatbot Processing

| Code                     | Purpose                                         |
| ------------------------ | ----------------------------------------------- |
| `process_user_message()` | Sends the user's message to the chatbot engine. |
| `response`               | Receives the AI-generated answer.               |
| `context`                | Updates the conversation history.               |


===



Chat with the chatbot!
Note that the system message includes detailed instructions about what the OrderBot should do.



Install Libraries

Imports

In [4]:
!pip install -q openai panel
from openai import OpenAI
from google.colab import userdata
import panel as pn

pn.extension()

Groq Client

In [5]:
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

Chat Function

In [6]:
def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=700
):

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

Chat Processing

In [7]:
def process_user_message(user_input, history):

    system_message = """
You are a helpful AI Service Assistant.

Answer clearly.

Be polite.

If you don't know, say so.
"""

    messages = history + [

        {
            "role":"system",
            "content":system_message
        },

        {
            "role":"user",
            "content":user_input
        }

    ]

    response = get_completion_from_messages(messages)

    history.append(
        {
            "role":"user",
            "content":user_input
        }
    )

    history.append(
        {
            "role":"assistant",
            "content":response
        }
    )

    return response, history

Panel Chat UI

In [10]:
chat_area = pn.Column()

history = []

inp = pn.widgets.TextInput(
    placeholder="Ask something..."
)

button = pn.widgets.Button(
    name="Send",
    button_type="primary"
)

def collect_messages(event):

    global history

    question = inp.value.strip()

    if question == "":
        return

    inp.value = ""

    # -------------------------
    # Exit Command
    # -------------------------
    if question.lower() in ["exit","quit","bye","close"]:

        chat_area.append(
            pn.Row(
                "🤖",
                pn.pane.Markdown(
                    """
### 👋 Session Closed

Thank you for chatting.

Type **Restart Runtime** or rerun the notebook to start a new conversation.
                    """,
                    width=650
                )
            )
        )

        history = []

        inp.disabled = True
        button.disabled = True

        return

    # -------------------------
    # Normal Chat
    # -------------------------

    try:

        answer, history = process_user_message(
            question,
            history
        )

    except Exception as e:

        answer = str(e)

    chat_area.append(

        pn.Row(
            "🧑",
            pn.pane.Markdown(question,width=650)
        )

    )

    chat_area.append(

        pn.Row(

            "🤖",

            pn.pane.Markdown(
                answer,
                width=650,
                styles={
                    "background":"#F6F6F6",
                    "padding":"10px",
                    "border-radius":"8px"
                }

            )

        )

    )
button.on_click(collect_messages)

Watcher(inst=Button(button_type='primary', color='primary', label='Send', name='Send'), cls=<class 'panel.widgets.button.Button'>, fn=<function collect_messages at 0x7a13f2229260>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

Remove warning message

In [12]:
import logging
import warnings

warnings.filterwarnings("ignore")

logging.getLogger("bokeh").setLevel(logging.ERROR)
logging.getLogger("panel").setLevel(logging.ERROR)

Dashboard

In [13]:
dashboard = pn.Column(

    "# 🤖 AI Service Assistant",

    inp,

    button,

    chat_area

)

dashboard

Column
    [0] Markdown(str)
    [1] TextInput(placeholder='Ask something...', value_input='what is python')
    [2] Button(button_type='primary', clicks=3, color='primary', label='Send', name='Send')
    [3] Column
        [0] Row
            [0] Markdown(str)
            [1] Markdown(str, width=650)
        [1] Row
            [0] Markdown(str)
            [1] Markdown(str, styles={'background': '#F6F6F6', ...}, width=650)
        [2] Row
            [0] Markdown(str)
            [1] Markdown(str, width=650)
        [3] Row
            [0] Markdown(str)
            [1] Markdown(str, styles={'background': '#F6F6F6', ...}, width=650)